# Building a Toy LLM from Scratch

This notebook walks through the core concepts behind Large Language Models (LLMs) by building one from scratch using PyTorch. We'll cover:

1. **Tokenization** - How text becomes numbers
2. **Embeddings** - How tokens become vectors
3. **Positional Encoding** - How order is preserved
4. **Self-Attention** - How tokens relate to each other
5. **Multi-Head Attention** - Attending to multiple patterns
6. **Transformer Block** - The building block of LLMs
7. **The Full Model** - Putting it all together
8. **Training** - Teaching the model language
9. **Text Generation** - Producing new text

---

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

# Set random seed for reproducibility
torch.manual_seed(42)

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## 1. Tokenization

LLMs don't understand text directly — they work with **tokens** (numbers). Tokenization is the process of converting text into a sequence of integers.

Real LLMs use sophisticated tokenizers like **BPE (Byte-Pair Encoding)** used in GPT, or **SentencePiece** used in LLaMA. These break words into subword units (e.g., "playing" → ["play", "ing"]).

For our toy model, we'll use a simple **character-level tokenizer** — each unique character gets its own token ID.

In [ ]:
# Our training corpus - a small text dataset
text = """The cat sat on the mat. The dog sat on the log.
A cat and a dog are friends. The cat likes fish.
The dog likes bones. They play in the park together.
The sun is warm and the sky is blue today.
Birds sing in the trees and flowers bloom in spring."""

print(f"Text length: {len(text)} characters")
print(f"First 80 chars: '{text[:80]}'")

In [ ]:
class CharTokenizer:
    """Simple character-level tokenizer.
    
    Maps each unique character to an integer ID.
    Real tokenizers (BPE, WordPiece) work on subword units for efficiency.
    """
    def __init__(self, text):
        # Get all unique characters (our vocabulary)
        self.chars = sorted(list(set(text)))
        self.vocab_size = len(self.chars)
        
        # Create mappings: character <-> integer
        self.char_to_id = {ch: i for i, ch in enumerate(self.chars)}
        self.id_to_char = {i: ch for i, ch in enumerate(self.chars)}
    
    def encode(self, text):
        """Convert text string to list of token IDs."""
        return [self.char_to_id[ch] for ch in text]
    
    def decode(self, ids):
        """Convert list of token IDs back to text string."""
        return ''.join([self.id_to_char[i] for i in ids])

# Create our tokenizer
tokenizer = CharTokenizer(text)
print(f"Vocabulary size: {tokenizer.vocab_size} unique characters")
print(f"Vocabulary: {tokenizer.chars}")
print()

# Demonstrate encoding/decoding
sample = "The cat"
encoded = tokenizer.encode(sample)
decoded = tokenizer.decode(encoded)
print(f"Original:  '{sample}'")
print(f"Encoded:   {encoded}")
print(f"Decoded:   '{decoded}'")

### Key Insight: Why Tokenization Matters

| Tokenizer Type | "playing" becomes | Vocab Size | Tradeoff |
|---|---|---|---|
| Character-level | ['p','l','a','y','i','n','g'] | ~100 | Long sequences, small vocab |
| BPE (GPT-2/3/4) | ['play', 'ing'] | ~50,000 | Good balance |
| Word-level | ['playing'] | ~100,000+ | Short sequences, huge vocab |

Real LLMs use BPE because it balances vocabulary size with sequence length.

---
## 2. Embeddings

Token IDs are just integers — they don't carry meaning. **Embeddings** convert each token ID into a dense vector (list of floating-point numbers) in a high-dimensional space.

The key idea: tokens with similar meanings end up with similar vectors. These vectors are **learned during training**.

- GPT-2 uses 768-dimensional embeddings
- GPT-3 uses 12,288-dimensional embeddings
- Our toy model will use 64-dimensional embeddings

In [ ]:
# Hyperparameters for our toy model
vocab_size = tokenizer.vocab_size
d_model = 64        # Embedding dimension (GPT-2: 768, GPT-3: 12288)
n_heads = 4         # Number of attention heads (GPT-2: 12)
n_layers = 2        # Number of transformer blocks (GPT-2: 12, GPT-3: 96)
context_length = 32 # Max sequence length (GPT-2: 1024, GPT-4: 128000)
d_ff = 128          # Feed-forward hidden dimension (usually 4x d_model)
dropout = 0.1

print(f"Model config:")
print(f"  Vocab size:      {vocab_size}")
print(f"  Embedding dim:   {d_model}")
print(f"  Attention heads: {n_heads}")
print(f"  Layers:          {n_layers}")
print(f"  Context length:  {context_length}")
print(f"  FF dimension:    {d_ff}")

In [ ]:
# Demonstrate how embedding works
embedding_layer = nn.Embedding(vocab_size, d_model)

# Encode "cat" and look at its embedding
sample_tokens = torch.tensor(tokenizer.encode("cat"))
sample_embeddings = embedding_layer(sample_tokens)

print(f"Input tokens: {sample_tokens} (characters: c, a, t)")
print(f"Embedding shape: {sample_embeddings.shape}  # (3 tokens, {d_model} dimensions each)")
print(f"\nEmbedding for 'c' (first 10 dims): {sample_embeddings[0, :10].detach().numpy().round(3)}")
print(f"Embedding for 'a' (first 10 dims): {sample_embeddings[1, :10].detach().numpy().round(3)}")
print(f"\nEach token is now a {d_model}-dimensional vector that the model can learn from!")

---
## 3. Positional Encoding

Unlike RNNs, Transformers process all tokens **in parallel** — they have no inherent notion of order. Without positional information, "the cat sat on the mat" and "mat the on sat cat the" would look the same!

**Positional encodings** add position information to embeddings. Two common approaches:
- **Sinusoidal** (original Transformer paper) — fixed mathematical patterns
- **Learned** (GPT-2/3) — trained position embeddings

We'll implement both and use learned positions (like GPT).

In [ ]:
# Sinusoidal Positional Encoding (from "Attention Is All You Need" paper)
def sinusoidal_positional_encoding(max_len, d_model):
    """Create fixed sinusoidal position encodings.
    
    Uses sin for even dimensions, cos for odd dimensions,
    with wavelengths forming a geometric progression.
    """
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    
    pe[:, 0::2] = torch.sin(position * div_term)  # Even dimensions
    pe[:, 1::2] = torch.cos(position * div_term)  # Odd dimensions
    return pe

# Visualize sinusoidal encodings
pe = sinusoidal_positional_encoding(context_length, d_model)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Heatmap of positional encodings
im = axes[0].imshow(pe.numpy(), aspect='auto', cmap='RdBu')
axes[0].set_xlabel('Embedding Dimension')
axes[0].set_ylabel('Position')
axes[0].set_title('Sinusoidal Positional Encodings')
plt.colorbar(im, ax=axes[0])

# Show specific dimensions across positions
for dim in [0, 1, 4, 8, 16]:
    axes[1].plot(pe[:, dim].numpy(), label=f'dim {dim}')
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Encoding Value')
axes[1].set_title('Encoding Values by Dimension')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Each position has a unique pattern. Lower dimensions oscillate faster.")
print("This allows the model to distinguish position AND compute relative distances.")

---
## 4. Self-Attention (The Core Innovation)

Self-attention is the **key mechanism** that makes Transformers powerful. It allows each token to "look at" every other token in the sequence and decide which ones are relevant.

### How it works:
For each token, we compute three vectors:
- **Query (Q)**: "What am I looking for?"
- **Key (K)**: "What do I contain?"
- **Value (V)**: "What information do I provide?"

### The attention formula:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

1. Compute similarity scores: $QK^T$ (dot product of queries and keys)
2. Scale by $\sqrt{d_k}$ to prevent large values
3. Apply softmax to get attention weights (sum to 1)
4. Multiply by values to get the output

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Compute scaled dot-product attention.
    
    Args:
        Q: Query tensor  [batch, heads, seq_len, d_k]
        K: Key tensor    [batch, heads, seq_len, d_k]
        V: Value tensor  [batch, heads, seq_len, d_k]
        mask: Optional causal mask to prevent attending to future tokens
    """
    d_k = Q.size(-1)
    
    # Step 1: Compute attention scores (how much each token attends to every other)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    # Step 2: Apply causal mask (for autoregressive models like GPT)
    # This prevents tokens from "seeing the future"
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Step 3: Softmax to get attention weights (probabilities that sum to 1)
    attention_weights = F.softmax(scores, dim=-1)
    
    # Step 4: Weighted sum of values
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

# Demonstrate attention with a simple example
print("=== Self-Attention Demonstration ===")
print()

# Simulate 4 tokens, each with 8 dimensions
seq_len, d_k = 4, 8
Q = torch.randn(1, 1, seq_len, d_k)
K = torch.randn(1, 1, seq_len, d_k)
V = torch.randn(1, 1, seq_len, d_k)

# Without mask (bidirectional - like BERT)
output, weights = scaled_dot_product_attention(Q, K, V)
print("Attention weights (no mask - each token sees ALL tokens):")
print(weights[0, 0].detach().numpy().round(3))
print()

# With causal mask (unidirectional - like GPT)
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)
output_masked, weights_masked = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
print("Attention weights (causal mask - tokens can only see PAST tokens):")
print(weights_masked[0, 0].detach().numpy().round(3))
print()
print("Notice: Row i only has non-zero values in columns 0..i (can't see future!)")

In [ ]:
# Visualize causal mask
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# The mask itself
axes[0].imshow(causal_mask[0, 0].numpy(), cmap='Blues')
axes[0].set_title('Causal Mask (1=allowed, 0=blocked)')
axes[0].set_xlabel('Key position (what we attend to)')
axes[0].set_ylabel('Query position (current token)')
for i in range(seq_len):
    for j in range(seq_len):
        axes[0].text(j, i, f'{int(causal_mask[0,0,i,j].item())}', ha='center', va='center')

# The attention weights
im = axes[1].imshow(weights_masked[0, 0].detach().numpy(), cmap='Reds')
axes[1].set_title('Resulting Attention Weights')
axes[1].set_xlabel('Key position')
axes[1].set_ylabel('Query position')
plt.colorbar(im, ax=axes[1])
for i in range(seq_len):
    for j in range(seq_len):
        axes[1].text(j, i, f'{weights_masked[0,0,i,j].item():.2f}', ha='center', va='center', fontsize=8)

plt.tight_layout()
plt.show()
print("\nGPT-style models use causal masking so they can only predict based on past context.")

---
## 5. Multi-Head Attention

Instead of one attention mechanism, we use **multiple heads** running in parallel. Each head can learn to focus on different types of relationships:
- Head 1 might learn syntactic relationships (subject-verb)
- Head 2 might learn semantic relationships (word meanings)
- Head 3 might learn positional patterns

The outputs are concatenated and projected back to the model dimension.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Self-Attention mechanism.
    
    Splits the embedding into multiple heads, applies attention independently,
    then concatenates and projects back.
    """
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # Dimension per head
        
        # Linear projections for Q, K, V (all heads computed at once)
        self.W_q = nn.Linear(d_model, d_model)  # Projects input to queries
        self.W_k = nn.Linear(d_model, d_model)  # Projects input to keys
        self.W_v = nn.Linear(d_model, d_model)  # Projects input to values
        self.W_o = nn.Linear(d_model, d_model)  # Output projection
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        
        # Project to Q, K, V
        Q = self.W_q(x)  # [batch, seq_len, d_model]
        K = self.W_k(x)
        V = self.W_v(x)
        
        # Reshape to [batch, n_heads, seq_len, d_k]
        Q = Q.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        
        # Apply scaled dot-product attention
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads and project
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.W_o(attn_output)
        
        return self.dropout(output), attn_weights

# Test multi-head attention
mha = MultiHeadAttention(d_model=64, n_heads=4)
test_input = torch.randn(2, 10, 64)  # batch=2, seq_len=10, d_model=64
output, weights = mha(test_input)
print(f"Input shape:            {test_input.shape}  # [batch, seq_len, d_model]")
print(f"Output shape:           {output.shape}  # [batch, seq_len, d_model] (same!)")
print(f"Attention weights shape: {weights.shape}  # [batch, n_heads, seq_len, seq_len]")
print(f"\nEach of the {mha.n_heads} heads operates on {mha.d_k} dimensions (64/4 = 16)")

---
## 6. Transformer Block

A Transformer block combines:
1. **Multi-Head Attention** (with residual connection + layer norm)
2. **Feed-Forward Network** (with residual connection + layer norm)

The feed-forward network (FFN) gives the model capacity to process information after attention has gathered context. It applies the same transformation to each position independently.

**Residual connections** (skip connections) help with training deep networks by allowing gradients to flow directly through the network.

**Layer normalization** stabilizes training by normalizing activations.

In [ ]:
class FeedForward(nn.Module):
    """Position-wise Feed-Forward Network.
    
    Two linear layers with a GELU activation in between.
    Expands to a larger dimension then projects back.
    This is where much of the "knowledge" is stored in LLMs.
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),     # Expand
            nn.GELU(),                     # Activation (smoother than ReLU)
            nn.Linear(d_ff, d_model),     # Project back
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """A single Transformer block (GPT uses 12-96 of these stacked).
    
    Architecture:
        x -> LayerNorm -> MultiHeadAttention -> + (residual) ->
          -> LayerNorm -> FeedForward -> + (residual) -> output
    """
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.ln1 = nn.LayerNorm(d_model)  # Pre-norm (GPT-2 style)
        self.ln2 = nn.LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        # Self-attention with residual connection
        attn_output, attn_weights = self.attention(self.ln1(x), mask)
        x = x + attn_output  # Residual connection
        
        # Feed-forward with residual connection
        ff_output = self.feed_forward(self.ln2(x))
        x = x + ff_output  # Residual connection
        
        return x, attn_weights

# Test a transformer block
block = TransformerBlock(d_model=64, n_heads=4, d_ff=128)
test_input = torch.randn(2, 10, 64)
output, weights = block(test_input)
print(f"Transformer Block: input {test_input.shape} -> output {output.shape}")
print(f"Shape is preserved through the block (residual connections!)")

---
## 7. The Full Toy LLM (GPT-style)

Now let's put everything together into a complete language model:

```
Input tokens
    ↓
Token Embeddings + Position Embeddings
    ↓
Transformer Block 1
    ↓
Transformer Block 2
    ↓
    ...  
    ↓
Layer Norm
    ↓
Linear (project to vocab size)
    ↓
Logits (probability of each token being next)
```

The model predicts the **next token** given all previous tokens. This is called **causal language modeling**.

In [ ]:
class ToyLLM(nn.Module):
    """A GPT-style language model built from scratch.
    
    This is a decoder-only transformer that predicts the next token
    given a sequence of previous tokens (autoregressive).
    """
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, context_length, dropout=0.1):
        super().__init__()
        
        # Token and position embeddings
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(context_length, d_model)  # Learned positions
        self.dropout = nn.Dropout(dropout)
        
        # Stack of transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        
        # Final layer norm and output projection
        self.ln_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Weight tying: share weights between token embedding and output projection
        # This is a common trick that improves performance and reduces parameters
        self.lm_head.weight = self.token_embedding.weight
        
        self.context_length = context_length
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, idx, targets=None):
        """
        Args:
            idx: Token indices [batch_size, seq_len]
            targets: Target token indices for computing loss
        Returns:
            logits: Predictions [batch_size, seq_len, vocab_size]
            loss: Cross-entropy loss (if targets provided)
        """
        batch_size, seq_len = idx.shape
        
        # Get token and position embeddings
        tok_emb = self.token_embedding(idx)  # [batch, seq_len, d_model]
        pos = torch.arange(0, seq_len, device=idx.device)
        pos_emb = self.position_embedding(pos)  # [seq_len, d_model]
        
        # Combine embeddings
        x = self.dropout(tok_emb + pos_emb)
        
        # Create causal mask
        mask = torch.tril(torch.ones(seq_len, seq_len, device=idx.device)).unsqueeze(0).unsqueeze(0)
        
        # Pass through transformer blocks
        for block in self.blocks:
            x, _ = block(x, mask)
        
        # Final layer norm and project to vocabulary
        x = self.ln_final(x)
        logits = self.lm_head(x)  # [batch, seq_len, vocab_size]
        
        # Compute loss if targets provided
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )
        
        return logits, loss

# Create our model!
model = ToyLLM(
    vocab_size=vocab_size,
    d_model=d_model,
    n_heads=n_heads,
    n_layers=n_layers,
    d_ff=d_ff,
    context_length=context_length,
    dropout=dropout
).to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"\nToy LLM created with {n_params:,} parameters")
print(f"(For comparison: GPT-2 has 124M, GPT-3 has 175B, GPT-4 estimated ~1.8T)")
print(f"\nModel architecture:")
print(model)

---
## 8. Training the Model

Training an LLM = teaching it to predict the next token.

Given the sequence `"The cat sat"`, the model learns:
- Given `"T"`, predict `"h"`
- Given `"Th"`, predict `"e"`  
- Given `"The"`, predict `" "`
- Given `"The "`, predict `"c"`
- ... and so on

The loss function is **cross-entropy** between predicted probabilities and actual next tokens.

In [ ]:
# Prepare training data
# Encode entire text into token IDs
data = torch.tensor(tokenizer.encode(text), dtype=torch.long)
print(f"Total tokens in dataset: {len(data)}")
print(f"First 50 tokens: {data[:50].tolist()}")
print(f"Decoded back:    '{tokenizer.decode(data[:50].tolist())}'")

In [ ]:
def get_batch(data, batch_size, context_length):
    """Generate a random batch of training examples.
    
    Each example is a sequence of `context_length` tokens (input)
    and the same sequence shifted by 1 (target).
    """
    # Random starting positions
    ix = torch.randint(len(data) - context_length - 1, (batch_size,))
    
    # Input: tokens at positions [i, i+context_length)
    x = torch.stack([data[i:i+context_length] for i in ix])
    # Target: tokens at positions [i+1, i+context_length+1) (shifted by 1)
    y = torch.stack([data[i+1:i+context_length+1] for i in ix])
    
    return x.to(device), y.to(device)

# Show an example batch
x_batch, y_batch = get_batch(data, batch_size=2, context_length=context_length)
print(f"Input batch shape:  {x_batch.shape}  # [batch_size, context_length]")
print(f"Target batch shape: {y_batch.shape}  # [batch_size, context_length]")
print()
print("Example (first sequence):")
print(f"  Input:  '{tokenizer.decode(x_batch[0].tolist())}'")
print(f"  Target: '{tokenizer.decode(y_batch[0].tolist())}'")
print()
print("Notice: target is input shifted by one position (next-token prediction!)")

In [ ]:
# Training loop
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)  # AdamW is standard for transformers

batch_size = 16
n_steps = 2000
eval_interval = 200

losses = []
print("Training the Toy LLM...")
print(f"{'Step':>6} | {'Loss':>8} | Sample Generation")
print("-" * 70)

model.train()
for step in range(n_steps):
    # Get a batch
    x_batch, y_batch = get_batch(data, batch_size, context_length)
    
    # Forward pass
    logits, loss = model(x_batch, y_batch)
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    # Print progress
    if step % eval_interval == 0 or step == n_steps - 1:
        # Generate a sample to see progress
        model.eval()
        with torch.no_grad():
            prompt = torch.tensor(tokenizer.encode("The "), dtype=torch.long).unsqueeze(0).to(device)
            generated = prompt[0].tolist()
            for _ in range(30):
                context = torch.tensor([generated[-context_length:]], dtype=torch.long).to(device)
                logits_gen, _ = model(context)
                probs = F.softmax(logits_gen[0, -1], dim=-1)
                next_token = torch.multinomial(probs, 1).item()
                generated.append(next_token)
            sample = tokenizer.decode(generated)
        model.train()
        print(f"{step:>6} | {loss.item():>8.4f} | '{sample[:50]}'")

print("\nTraining complete!")

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(losses, alpha=0.3, color='blue', label='Raw loss')
# Smoothed loss
window = 50
smoothed = [sum(losses[max(0,i-window):i+1])/min(i+1, window) for i in range(len(losses))]
plt.plot(smoothed, color='red', linewidth=2, label=f'Smoothed (window={window})')
plt.xlabel('Training Step')
plt.ylabel('Cross-Entropy Loss')
plt.title('Training Loss Over Time')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Initial loss: {losses[0]:.4f} (random: -ln(1/{vocab_size}) = {math.log(vocab_size):.4f})")
print(f"Final loss:   {losses[-1]:.4f}")
print(f"\nThe model went from random guessing to learning patterns in the text!")

---
## 9. Text Generation (Inference)

Now let's use our trained model to generate text! The process:
1. Start with a **prompt** (seed text)
2. Feed it through the model to get probabilities for the next token
3. **Sample** from those probabilities (or take the most likely token)
4. Append the new token and repeat

### Temperature
Controls randomness in generation:
- **Low temperature (0.1-0.5)**: More deterministic, repetitive
- **Temperature = 1.0**: Normal sampling from learned distribution
- **High temperature (1.5+)**: More random, creative, but can be nonsensical

### Top-k Sampling
Only consider the top k most likely tokens (filters out unlikely noise).

In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=100, temperature=1.0, top_k=None):
    """Generate text from the model.
    
    Args:
        prompt: Starting text string
        max_new_tokens: How many new tokens to generate
        temperature: Controls randomness (lower = more deterministic)
        top_k: If set, only sample from top k most likely tokens
    """
    model.eval()
    
    # Encode the prompt
    tokens = tokenizer.encode(prompt)
    
    for _ in range(max_new_tokens):
        # Crop to context length
        context = tokens[-model.context_length:]
        x = torch.tensor([context], dtype=torch.long).to(device)
        
        # Get predictions
        logits, _ = model(x)
        logits = logits[0, -1, :]  # Last position predictions
        
        # Apply temperature
        logits = logits / temperature
        
        # Apply top-k filtering
        if top_k is not None:
            values, _ = torch.topk(logits, top_k)
            logits[logits < values[-1]] = float('-inf')
        
        # Convert to probabilities and sample
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, 1).item()
        
        tokens.append(next_token)
    
    return tokenizer.decode(tokens)

# Generate with different settings
print("=" * 70)
print("TEXT GENERATION EXAMPLES")
print("=" * 70)

prompts = ["The cat", "The dog", "Birds "]

for prompt in prompts:
    print(f"\nPrompt: '{prompt}'")
    print(f"  Temperature=0.5: '{generate(model, tokenizer, prompt, max_new_tokens=60, temperature=0.5)}'")
    print(f"  Temperature=1.0: '{generate(model, tokenizer, prompt, max_new_tokens=60, temperature=1.0)}'")
    print(f"  Temperature=1.5: '{generate(model, tokenizer, prompt, max_new_tokens=60, temperature=1.5)}'")

In [ ]:
# Visualize attention patterns of our trained model
print("=" * 70)
print("ATTENTION VISUALIZATION")
print("=" * 70)

model.eval()
sample_text = "The cat sat on"
sample_tokens = torch.tensor([tokenizer.encode(sample_text)], dtype=torch.long).to(device)

# Forward pass to get attention weights
with torch.no_grad():
    # Get attention from first block
    tok_emb = model.token_embedding(sample_tokens)
    pos = torch.arange(sample_tokens.size(1), device=device)
    pos_emb = model.position_embedding(pos)
    x = model.dropout(tok_emb + pos_emb)
    mask = torch.tril(torch.ones(sample_tokens.size(1), sample_tokens.size(1), device=device)).unsqueeze(0).unsqueeze(0)
    _, attn_weights = model.blocks[0](x, mask)

# Plot attention for each head
chars = list(sample_text)
fig, axes = plt.subplots(1, n_heads, figsize=(16, 4))

for head in range(n_heads):
    ax = axes[head]
    weights_np = attn_weights[0, head].cpu().numpy()
    im = ax.imshow(weights_np, cmap='Blues')
    ax.set_xticks(range(len(chars)))
    ax.set_yticks(range(len(chars)))
    ax.set_xticklabels(chars, fontsize=8)
    ax.set_yticklabels(chars, fontsize=8)
    ax.set_title(f'Head {head+1}')
    ax.set_xlabel('Attends to')
    if head == 0:
        ax.set_ylabel('From token')

plt.suptitle(f'Attention Patterns for "{sample_text}"', fontsize=12)
plt.tight_layout()
plt.show()
print("\nEach head learns different attention patterns!")
print("Some heads might focus on adjacent tokens, others on specific characters.")

---
## 10. Key Concepts Summary

### What We Built vs Real LLMs

| Aspect | Our Toy LLM | GPT-4 (estimated) |
|--------|-------------|-------------------|
| Parameters | ~30K | ~1.8 Trillion |
| Layers | 2 | 120+ |
| Context | 32 tokens | 128,000 tokens |
| Training data | ~250 chars | Trillions of tokens |
| Training time | Seconds | Months on thousands of GPUs |
| Tokenizer | Character | BPE (~100K vocab) |

### The Key Ideas That Make LLMs Work

1. **Scale**: More parameters + more data = emergent capabilities
2. **Self-Attention**: Allows modeling long-range dependencies
3. **Next-Token Prediction**: Simple objective, surprisingly powerful
4. **Transfer Learning**: Pre-train on general text, fine-tune for tasks

### Additional Topics in Real LLMs (not covered here)

- **RLHF (Reinforcement Learning from Human Feedback)**: Aligning model outputs with human preferences
- **KV Cache**: Speeding up generation by caching key/value pairs
- **Rotary Position Embeddings (RoPE)**: Better position encoding used in modern models
- **Flash Attention**: Memory-efficient attention computation
- **Mixture of Experts (MoE)**: Activating only a subset of parameters per token
- **Quantization**: Reducing model size (e.g., 16-bit → 4-bit)
- **LoRA/QLoRA**: Efficient fine-tuning with low-rank adaptations

In [ ]:
# Final interactive demo - try your own prompts!
print("=" * 70)
print("INTERACTIVE GENERATION")
print("=" * 70)
print("\nNote: Our model was trained on a tiny corpus, so it can only generate")
print("text similar to what it saw. Try prompts that start like the training text.")
print()

test_prompts = [
    "The ",
    "A cat",
    "They ",
    "The sun",
]

for prompt in test_prompts:
    result = generate(model, tokenizer, prompt, max_new_tokens=80, temperature=0.7, top_k=10)
    print(f"Prompt: '{prompt}' → '{result}'")
    print()

---
## Summary: How an LLM Processes a Prompt

```
"The cat sat" (input text)
       ↓
[Tokenizer] → [24, 8, 5, 1, 3, 2, 20, 1, 19, 2, 20]  (token IDs)
       ↓
[Embedding Layer] → 11 vectors of dimension d_model
       ↓
[+ Position Embeddings] → adds position information
       ↓
[Transformer Block 1]
  ├── Self-Attention: tokens gather context from relevant other tokens
  └── Feed-Forward: process the gathered information
       ↓
[Transformer Block 2]
  ├── Self-Attention: higher-level patterns
  └── Feed-Forward: more processing
       ↓
[Output Layer] → probability distribution over vocabulary
       ↓
[Sampling] → pick next token (e.g., " " or "o" or "s")
       ↓
Append token and repeat!
```

That's it! The fundamental architecture behind ChatGPT, Claude, LLaMA, and all modern LLMs.